# Признаки cookie

Каждая cookie превращается в одну строку. Train и test проходят одну и ту же обработку, а `target` используется только после построения признаков. На выходе сохраняются две матрицы в `features/` и список колонок для модели.

События берём из полуинтервала `[window_start_ts, window_end_ts)`: левый край входит, правый нет. Даты окна сохраняются для временной проверки, но в модель не попадают.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    nested = ROOT / "bot_detection_challenge"
    if (nested / "data" / "train.csv").exists():
        ROOT = nested
    else:
        raise FileNotFoundError("")

DATA_DIR = ROOT / "data"
FEATURES_DIR = ROOT / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

ALL_EVENTS = [
    "item_view",
    "search_results_view",
    "photo_swipe",
    "contact_phone_show",
    "contact_chat_open",
    "contact_message_sent",
    "favorite_add",
    "seller_page_view",
    "login",
]
META_COLS = ["cookie_id", "cookie_created_at", "window_start_ts", "window_end_ts"]
DATE_COLS = ["cookie_created_at", "window_start_ts", "window_end_ts"]
print(ROOT)


/Users/a123123/avito_test/bot_detection_challenge


## 1. Загрузка, дубли и окно

Полный дубль строки в `events` — повтор одного клика. Удаляем такие строки во всей таблице до фильтрации окна, частот и пауз: иначе один клик посчитается дважды, а пауза между копиями станет нулевой. Разные события с одинаковым временем не склеиваем.

В историю cookie попадает полуинтервал `[window_start_ts, window_end_ts)`. Событие ровно на границе следующих суток относится только к следующему окну.

In [5]:
def drop_full_duplicates(df, name):
    before = len(df)
    out = df.drop_duplicates().reset_index(drop=True)
    print(f"{name}: удалено {before - len(out)} полных дублей, осталось {len(out)}")
    return out


def events_in_window(events, meta):
    ev = events.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id",
        how="inner",
        validate="many_to_one",
    )
    # [window_start_ts, window_end_ts): правый край не входит
    mask = (ev["event_ts"] >= ev["window_start_ts"]) & (ev["event_ts"] < ev["window_end_ts"])
    return ev.loc[mask].drop(columns=["window_start_ts", "window_end_ts"]).copy()


train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=DATE_COLS)
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=DATE_COLS)
events = pd.read_csv(DATA_DIR / "events.csv.gz", parse_dates=["event_ts"])

train = drop_full_duplicates(train, "train")
test = drop_full_duplicates(test, "test")
events = drop_full_duplicates(events, "events")

assert train["cookie_id"].is_unique and test["cookie_id"].is_unique
assert set(train["cookie_id"]).isdisjoint(set(test["cookie_id"]))
assert train["window_start_ts"].max() < test["window_start_ts"].min()

ev_train = events_in_window(events, train)
ev_test = events_in_window(events, test)
print(f"событий в окне: train {len(ev_train):,}, test {len(ev_test):,}")

train: удалено 0 полных дублей, осталось 11091
test: удалено 0 полных дублей, осталось 4909
events: удалено 4863 полных дублей, осталось 324042
событий в окне: train 195,484, test 88,349


## 2. Маркеры на каждом событии

Перед расчётом пауз сортируем события по `cookie_id` и времени. Для события сохраняем паузу до предыдущего действия, час, тип перехода и признаки типа события.

Для событий ночным считаем интервал 01:00–06:00: часы 1, 2, 3, 4 и 5. Флаг часа создания cookie сейчас использует часы 00:00–06:00 и поэтому включает полночь; это отдельное определение, которое нужно учитывать при чтении признаков.

Платформу приводим к нижнему регистру. В модель идут не сырые строки `user_agent`, а три флага: headless-клиент, скриптовая библиотека и приложение Avito. Отдельно сохраняем признаки desktop/mobile.

In [6]:
def prepare_events(ev):
    ev = ev.sort_values(["cookie_id", "event_ts"], kind="mergesort").copy()
    ev["platform_clean"] = ev["platform"].astype(str).str.lower()
    ua = ev["user_agent"].astype(str).str.lower()

    ev["is_crawler_lib"] = ua.str.contains(
        r"scrapy|curl|requests|urllib|http-client|fetch|python", regex=True
    ).astype(np.int8)
    ev["is_headless"] = ua.str.contains(
        r"headlesschrome|phantomjs|selenium|playwright", regex=True
    ).astype(np.int8)
    ev["is_avito_app"] = ua.str.contains(r"okhttp|avito/", regex=True).astype(np.int8)
    ev["is_desktop_plat"] = ev["platform_clean"].isin(["desktop", "web"]).astype(np.int8)
    ev["is_mobile_plat"] = ev["platform_clean"].isin(["android", "ios", "iphone"]).astype(np.int8)

    grouped = ev.groupby("cookie_id", sort=False)
    ev["dt"] = grouped["event_ts"].diff().dt.total_seconds()
    ev["pos"] = grouped.cumcount()
    ev["has_pointer"] = ev["pointer_x"].notna().astype(np.int8)
    ev["prev_event_name"] = grouped["event_name"].shift(1)
    ev["is_night"] = ev["event_ts"].dt.hour.isin([1, 2, 3, 4, 5]).astype(np.int8)
    ev["hour"] = ev["event_ts"].dt.hour.astype(np.int8)

    for name in ALL_EVENTS:
        ev[f"ev_{name}"] = (ev["event_name"] == name).astype(np.int8)

    ev["is_private_seller"] = (ev["seller_type"] == "private").astype(np.int8)
    ev["is_pro_seller"] = (ev["seller_type"] == "pro").astype(np.int8)

    prev, cur = ev["prev_event_name"], ev["event_name"]
    ev["trans_item_to_photo"] = (prev.eq("item_view") & cur.eq("photo_swipe")).astype(np.int8)
    ev["trans_item_to_phone"] = (prev.eq("item_view") & cur.eq("contact_phone_show")).astype(np.int8)
    ev["trans_item_to_item"] = (prev.eq("item_view") & cur.eq("item_view")).astype(np.int8)
    ev["trans_search_to_search"] = (
        prev.eq("search_results_view") & cur.eq("search_results_view")
    ).astype(np.int8)
    ev["trans_photo_to_photo"] = (prev.eq("photo_swipe") & cur.eq("photo_swipe")).astype(np.int8)

    first_login = ev.loc[ev["event_name"].eq("login")].groupby("cookie_id")["pos"].min()
    ev["after_login"] = ev["pos"].gt(ev["cookie_id"].map(first_login)).astype(np.float32)

    changed = ev["event_name"].ne(grouped["event_name"].shift())
    ev["_run"] = changed.groupby(ev["cookie_id"], sort=False).cumsum()
    return ev


def ratio_on_gaps(dt, cookie, threshold):
    valid = dt.notna()
    return dt.lt(threshold).where(valid).groupby(cookie).mean()


def page_arith_run(ev):
    """Самая длинная серия страниц поиска с шагом ровно +1."""
    search = (
        ev.dropna(subset=["search_page"])[["cookie_id", "search_page"]]
        .sort_values(["cookie_id", "search_page"], kind="mergesort")
        .copy()
    )
    if search.empty:
        return pd.Series(dtype=float)
    search["inc"] = search.groupby("cookie_id", sort=False)["search_page"].diff().eq(1)
    search["_chg"] = search["inc"].ne(search.groupby("cookie_id", sort=False)["inc"].shift())
    search["_run"] = search.groupby("cookie_id", sort=False)["_chg"].cumsum()
    run_len = search.groupby(["cookie_id", "_run", "inc"], sort=False).size()
    if True not in run_len.index.get_level_values("inc"):
        return pd.Series(0.0, index=search["cookie_id"].unique())
    return run_len.xs(True, level="inc").groupby(level=0).max()


def cursor_speed(ev):
    """Медианная скорость курсора, px/с. Берутся только шаги 0 < dt < 60."""
    pts = ev.dropna(subset=["pointer_x", "pointer_y"]).sort_values(
        ["cookie_id", "event_ts"], kind="mergesort"
    )
    if pts.empty:
        return pd.Series(dtype=float)
    grouped = pts.groupby("cookie_id", sort=False)
    dist = np.hypot(grouped["pointer_x"].diff(), grouped["pointer_y"].diff())
    dt = grouped["event_ts"].diff().dt.total_seconds()
    speed = dist / dt.where((dt > 0) & (dt < 60))
    return speed.groupby(pts["cookie_id"]).median()


## 3. Сводка на cookie

Одна cookie — одна строка. Пропуск координаты или страницы поиска оставляем пропуском: это означает, что поле не применялось. Ноль используем только там, где отсутствие действия имеет смысл, например для флага глубокой выдачи, если поиска не было.

### Ритм

Считаем число событий, длительность активности, квантили пауз и доли пауз короче 30, 1 и 0.3 секунды. `dt_std` для cookie с одним событием остаётся пропуском: без второй точки разброс пауз не определён. `night_events_ratio` использует интервал 01:00–06:00.

### Курсор

Для cookie с координатами считаем разброс по x и y, число уникальных точек и медианную скорость. Если координат нет, разброс и скорость остаются пропусками, а не превращаются в нулевое движение.

### Каталог

Считаем уникальные объявления, категории, локации и запросы, глубину выдачи, средний скачок страницы и максимальную последовательность страниц с шагом +1. Нулевые флаги глубины означают, что cookie не дошла до соответствующей страницы или не открывала поиск.

### Воронка

Сохраняем число и долю каждого типа события, переходы между типами, энтропию последовательности, длину повторяющейся серии и показатели действий после логина. Отношения с поиском используют ноль только как «поиска не было».

### Клиент

Сохраняем флаги headless, скриптовой библиотеки и приложения, признаки desktop/mobile и число разных `user_agent` за окно. Несколько агентов у одной cookie отмечаем отдельно.

### Возраст cookie

Возраст считается от `cookie_created_at` до начала окна. Дополнительно сохраняем возраст в часах и днях, логарифм возраста, признаки свежей cookie и час создания. Ночным часом создания считаем 01:00–06:00; `is_created_cron_peak` не связываем с выводами EDA и используем только как отдельный технический признак.

### Ранг внутри дня

Ранги считаются среди cookie с той же датой начала окна. Ранжируются `n_events`, `location_nunique`, `item_nunique`, `search_page_max`, `events_per_sec` и `dt_median`: это основные показатели активности, разнообразия и темпа. Популярность объявления ранжируется отдельно после каузального расчёта.

In [7]:
def finalize_frame(res):
    res = res.copy()
    for col in res.columns:
        if col in META_COLS:
            continue
        if res[col].dtype == object:
            res[col] = pd.to_numeric(res[col], errors="coerce")
    numeric = res.select_dtypes(include="number").columns
    res[numeric] = res[numeric].replace([np.inf, -np.inf], np.nan)
    return res


def extract_cookie_features(ev_df, meta_df):
    ev = prepare_events(ev_df)
    cookie = ev["cookie_id"]
    grouped = ev.groupby("cookie_id", sort=False)
    dt = ev["dt"]

    feat = pd.DataFrame(index=grouped.size().index)
    feat.index.name = "cookie_id"
    feat["n_events"] = grouped.size()
    feat["active_duration_sec"] = (grouped["event_ts"].max() - grouped["event_ts"].min()).dt.total_seconds()
    feat["events_per_sec"] = feat["n_events"] / (feat["active_duration_sec"] + 1.0)

    feat["dt_mean"] = grouped["dt"].mean()
    feat["dt_std"] = grouped["dt"].std()
    feat["dt_median"] = grouped["dt"].median()
    feat["dt_min"] = grouped["dt"].min()
    feat["dt_q25"] = grouped["dt"].quantile(0.25)
    feat["dt_q75"] = grouped["dt"].quantile(0.75)
    feat["dt_iqr"] = feat["dt_q75"] - feat["dt_q25"]
    feat["dt_fast_ratio_30s"] = ratio_on_gaps(dt, cookie, 30.0).astype(float)
    feat["dt_fast_ratio_1s"] = ratio_on_gaps(dt, cookie, 1.0).astype(float)
    feat["dt_ultrafast_ratio_03s"] = ratio_on_gaps(dt, cookie, 0.3).astype(float)
    valid_dt = dt.notna()
    feat["dt_int_bias_1s"] = (dt.sub(1.0).abs().lt(0.05)).where(valid_dt).groupby(cookie).mean().astype(float)
    feat = feat.copy()

    feat["night_events_cnt"] = grouped["is_night"].sum()
    feat["night_events_ratio"] = feat["night_events_cnt"] / feat["n_events"]
    feat["active_hours_nunique"] = ev["hour"].groupby(cookie).nunique()

    feat["pointer_ratio"] = grouped["has_pointer"].mean()
    feat["pointer_x_std"] = grouped["pointer_x"].std()
    feat["pointer_y_std"] = grouped["pointer_y"].std()
    pointed = ev.dropna(subset=["pointer_x", "pointer_y"])
    if pointed.empty:
        feat["pointer_unique_pts"] = 0.0
    else:
        feat["pointer_unique_pts"] = (
            pointed.groupby(["cookie_id", "pointer_x", "pointer_y"], sort=False).size().groupby(level=0).size()
        )
    feat["pointer_speed_median"] = cursor_speed(ev)
    feat = feat.copy()

    feat["search_page_max"] = grouped["search_page"].max()
    feat["search_page_mean"] = grouped["search_page"].mean()
    feat["search_page_std"] = grouped["search_page"].std()
    feat["search_page_nunique"] = grouped["search_page"].nunique()
    feat["is_deep_search_5"] = (feat["search_page_max"].fillna(0) > 5).astype(np.int8)
    feat["is_deep_search_10"] = (feat["search_page_max"].fillna(0) > 10).astype(np.int8)
    feat["is_deep_search_20"] = (feat["search_page_max"].fillna(0) >= 20).astype(np.int8)

    feat["item_nunique"] = grouped["item_id"].nunique()
    feat["category_nunique"] = grouped["item_category"].nunique()
    feat["location_nunique"] = grouped["item_location"].nunique()
    feat["is_multi_location"] = (feat["location_nunique"] > 5).astype(np.int8)
    feat["query_nunique"] = grouped["search_query"].nunique()
    feat["items_per_query"] = feat["item_nunique"] / (feat["query_nunique"] + 1.0)
    feat = feat.copy()

    feat["seller_private_cnt"] = grouped["is_private_seller"].sum()
    feat["seller_pro_cnt"] = grouped["is_pro_seller"].sum()
    feat["seller_private_ratio"] = feat["seller_private_cnt"] / (feat["item_nunique"] + 1e-5)
    feat["seller_pro_ratio"] = feat["seller_pro_cnt"] / (feat["item_nunique"] + 1e-5)

    feat["is_crawler_lib_max"] = grouped["is_crawler_lib"].max()
    feat["is_headless_max"] = grouped["is_headless"].max()
    feat["is_avito_app_max"] = grouped["is_avito_app"].max()
    feat["is_desktop_max"] = grouped["is_desktop_plat"].max()
    feat["is_mobile_max"] = grouped["is_mobile_plat"].max()
    feat["desktop_no_pointer"] = ((feat["is_desktop_max"] == 1) & (feat["pointer_ratio"] == 0)).astype(np.int8)
    feat["ua_nunique"] = grouped["user_agent"].nunique()
    feat = feat.copy()

    for name in ALL_EVENTS:
        feat[f"cnt_{name}"] = grouped[f"ev_{name}"].sum()
        feat[f"ratio_{name}"] = feat[f"cnt_{name}"] / feat["n_events"]

    feat["photo_to_item_ratio"] = feat["cnt_photo_swipe"] / (feat["cnt_item_view"] + 1.0)
    feat["phone_to_item_ratio"] = feat["cnt_contact_phone_show"] / (feat["cnt_item_view"] + 1.0)
    feat["chat_to_item_ratio"] = feat["cnt_contact_chat_open"] / (feat["cnt_item_view"] + 1.0)
    feat["item_repeat_ratio"] = 1.0 - (feat["item_nunique"] / (feat["cnt_item_view"] + 1e-5))
    feat["total_contacts"] = (
        feat["cnt_contact_phone_show"] + feat["cnt_contact_chat_open"] + feat["cnt_contact_message_sent"]
    )
    feat["total_contacts_ratio"] = feat["total_contacts"] / feat["n_events"]

    feat["trans_item_to_photo_cnt"] = grouped["trans_item_to_photo"].sum()
    feat["trans_item_to_photo_ratio"] = feat["trans_item_to_photo_cnt"] / (feat["cnt_item_view"] + 1.0)
    feat["trans_item_to_phone_cnt"] = grouped["trans_item_to_phone"].sum()
    feat["trans_item_to_phone_ratio"] = feat["trans_item_to_phone_cnt"] / (feat["cnt_item_view"] + 1.0)
    feat["trans_item_to_item_cnt"] = grouped["trans_item_to_item"].sum()
    feat["trans_search_to_search_cnt"] = grouped["trans_search_to_search"].sum()
    feat["trans_photo_to_photo_cnt"] = grouped["trans_photo_to_photo"].sum()

    feat["page_arith_run"] = page_arith_run(ev).reindex(feat.index).fillna(0.0)
    feat["items_per_search"] = feat["item_nunique"] / (feat["cnt_search_results_view"] + 1.0)
    feat["fav_per_search"] = feat["cnt_favorite_add"] / (feat["cnt_search_results_view"] + 1.0)
    feat["contact_per_search"] = feat["total_contacts"] / (feat["cnt_search_results_view"] + 1.0)
    feat["events_after_login_share"] = grouped["after_login"].mean().fillna(0.0)

    run_sizes = ev.groupby(["cookie_id", "_run"], sort=False).size()
    feat["run_len_max"] = run_sizes.groupby(level=0).max()

    shares = np.column_stack([feat[f"ratio_{name}"].fillna(0.0).to_numpy() for name in ALL_EVENTS])
    shares = np.clip(shares, 1e-12, 1.0)
    feat["event_entropy"] = -(shares * np.log(shares)).sum(axis=1)
    feat = feat.copy()

    search_ev = ev.dropna(subset=["search_page"])
    if search_ev.empty:
        feat["page_jump_mean"] = 0.0
        feat["page_skip_share"] = 0.0
        feat["page_next_share"] = 0.0
    else:
        jump = search_ev.groupby("cookie_id", sort=False)["search_page"].diff()
        abs_jump = jump.abs()
        feat["page_jump_mean"] = abs_jump.groupby(search_ev["cookie_id"]).mean()
        feat["page_skip_share"] = abs_jump.gt(1).groupby(search_ev["cookie_id"]).mean()
        feat["page_next_share"] = jump.eq(1).groupby(search_ev["cookie_id"]).mean()
    feat["no_text_search"] = (feat["cnt_search_results_view"] == 0).astype(np.int8)

    res = meta_df[META_COLS].merge(feat.copy().reset_index(), on="cookie_id", how="left")
    age_hours = (res["window_start_ts"] - res["cookie_created_at"]).dt.total_seconds() / 3600.0
    res["cookie_age_hours"] = age_hours
    res["cookie_age_days"] = age_hours / 24.0
    res["log_cookie_age_hours"] = np.log1p(np.maximum(age_hours, 0))
    res["is_fresh_cookie_24h"] = (age_hours < 24.0).astype(np.int8)
    res["is_fresh_cookie_1h"] = (age_hours < 1.0).astype(np.int8)
    res["created_hour"] = res["cookie_created_at"].dt.hour.astype(np.int8)
    res["is_created_at_night"] = res["created_hour"].isin([0, 1, 2, 3, 4, 5]).astype(np.int8)
    res["is_created_cron_peak"] = res["created_hour"].isin([10, 15, 19, 21]).astype(np.int8)
    res["item_uniq_x_new"] = (res["item_repeat_ratio"] * (res["cookie_age_days"] < 7.0).astype(float)).fillna(0.0)
    res["traverse_intensity"] = (
        (1.0 - (res["cnt_login"].fillna(0) > 0).astype(float))
        * np.log1p(res["search_page_max"].fillna(0))
        * np.log1p(res["location_nunique"].fillna(0))
        / (res["cookie_age_days"].clip(lower=0) + 1.0)
    ).fillna(0.0)
    res["has_login"] = (res["cnt_login"].fillna(0) > 0).astype(np.int8)
    res["login_x_locations"] = res["has_login"] * res["location_nunique"].fillna(0)
    res["login_x_events"] = res["has_login"] * res["n_events"].fillna(0)

    res["_day"] = res["window_start_ts"].dt.normalize()
    for col in ["n_events", "location_nunique", "item_nunique", "search_page_max", "events_per_sec", "dt_median"]:
        res[f"peer_{col}"] = res.groupby("_day")[col].rank(pct=True)
    return finalize_frame(res.drop(columns=["_day"]))


## 4. Популярность объявления без будущего

Для объявления считаем, сколько cookie с окном не позже текущего дня его уже открывали. Дни после окна не входят: иначе проверка подсмотрит, что объявление станет популярным завтра. На test в сумму входит весь train, потому что эти окна уже в прошлом.

В модель кладём не сырое число просмотров, а `log1p` среднего и минимума по объявлениям cookie и ранг этой величины среди cookie того же дня. Среднее говорит, насколько протоптанные объявления смотрит cookie, минимум — смотрела ли она что-то, что до неё почти никто не открывал.


In [8]:
def attach_causal_item_popularity(train_features, test_features, train_events, test_events, train_meta, test_meta):
    # день D видит только окна не позже D. На test в сумму входит весь train.
    frames = []
    for events_df, meta in ((train_events, train_meta), (test_events, test_meta)):
        day = meta.set_index("cookie_id")["window_start_ts"].dt.normalize()
        part = events_df.dropna(subset=["item_id"])[["cookie_id", "item_id"]].copy()
        part["day"] = part["cookie_id"].map(day)
        frames.append(part.dropna(subset=["day"]))
    trips = pd.concat(frames, ignore_index=True).drop_duplicates()

    days = np.array(sorted(trips["day"].unique()))
    daily = trips.groupby(["item_id", "day"], sort=False)["cookie_id"].nunique().rename("n").reset_index()
    wide = daily.pivot(index="item_id", columns="day", values="n").reindex(columns=days).fillna(0.0)
    lookup = wide.cumsum(axis=1).stack().rename("pop").reset_index()
    lookup.columns = ["item_id", "day", "pop"]

    seen = trips.merge(lookup, on=["item_id", "day"], how="left")
    agg = seen.groupby("cookie_id", sort=False)["pop"].agg(item_pop_mean="mean", item_pop_min="min")
    agg["item_pop_mean"] = np.log1p(agg["item_pop_mean"])
    agg["item_pop_min"] = np.log1p(agg["item_pop_min"])

    def _apply(features):
        out = features.drop(
            columns=[c for c in ("item_pop_mean", "item_pop_min", "peer_item_pop_mean") if c in features.columns]
        )
        out = out.merge(agg, left_on="cookie_id", right_index=True, how="left")
        out["item_pop_mean"] = out["item_pop_mean"].fillna(0.0)
        out["item_pop_min"] = out["item_pop_min"].fillna(0.0)
        day = out["window_start_ts"].dt.normalize()
        peer = out["item_pop_mean"].groupby(day).rank(pct=True).rename("peer_item_pop_mean")
        out = pd.concat([out, peer], axis=1)
        return finalize_frame(out)

    return _apply(train_features), _apply(test_features)


train_fe = extract_cookie_features(ev_train, train)
test_fe = extract_cookie_features(ev_test, test)
train_fe, test_fe = attach_causal_item_popularity(train_fe, test_fe, ev_train, ev_test, train, test)
target = pd.Series(
    train.set_index("cookie_id").loc[train_fe["cookie_id"], "target"].to_numpy(),
    index=train_fe.index,
    name="target",
)
train_fe = pd.concat([train_fe, target], axis=1)

feature_cols = [c for c in test_fe.columns if c not in META_COLS]
assert feature_cols == [c for c in train_fe.columns if c not in META_COLS + ["target"]]
assert train_fe["cookie_id"].tolist() == train["cookie_id"].tolist()
assert test_fe["cookie_id"].tolist() == test["cookie_id"].tolist()
print(f"train {train_fe.shape}, test {test_fe.shape}, признаков {len(feature_cols)}")
print(f"доля пропусков: {train_fe[feature_cols].isna().mean().mean():.2%}")
print("порядок cookie совпадает с csv")
print(feature_cols)


train (11091, 115), test (4909, 114), признаков 110
доля пропусков: 3.44%
порядок cookie совпадает с csv
['n_events', 'active_duration_sec', 'events_per_sec', 'dt_mean', 'dt_std', 'dt_median', 'dt_min', 'dt_q25', 'dt_q75', 'dt_iqr', 'dt_fast_ratio_30s', 'dt_fast_ratio_1s', 'dt_ultrafast_ratio_03s', 'dt_int_bias_1s', 'night_events_cnt', 'night_events_ratio', 'active_hours_nunique', 'pointer_ratio', 'pointer_x_std', 'pointer_y_std', 'pointer_unique_pts', 'pointer_speed_median', 'search_page_max', 'search_page_mean', 'search_page_std', 'search_page_nunique', 'is_deep_search_5', 'is_deep_search_10', 'is_deep_search_20', 'item_nunique', 'category_nunique', 'location_nunique', 'is_multi_location', 'query_nunique', 'items_per_query', 'seller_private_cnt', 'seller_pro_cnt', 'seller_private_ratio', 'seller_pro_ratio', 'is_crawler_lib_max', 'is_headless_max', 'is_avito_app_max', 'is_desktop_max', 'is_mobile_max', 'desktop_no_pointer', 'ua_nunique', 'cnt_item_view', 'ratio_item_view', 'cnt_search

## 5. Сохранение

Пишем `features/train_features.parquet`, `features/test_features.parquet` и `features/feature_columns.json`. В списке колонок нет идентификатора, дат и метки. Проверяем, что строки идут в том же порядке, что в исходных csv: файл ответов потом собирается без перестановки.


In [9]:

for stale in ("train_features.csv", "test_features.csv"):
    path = FEATURES_DIR / stale
    if path.exists():
        path.unlink()

train_fe.to_parquet(FEATURES_DIR / "train_features.parquet", index=False)
test_fe.to_parquet(FEATURES_DIR / "test_features.parquet", index=False)
pd.Series(feature_cols, name="feature").to_json(
    FEATURES_DIR / "feature_columns.json", orient="values", indent=2, force_ascii=False
)
print(f"сохранено в {FEATURES_DIR}")


сохранено в /Users/a123123/avito_test/bot_detection_challenge/features
